# Shopee Image Moderation — NSFW + QR/Barcode

Detect **NSFW** (khỏa thân, bikini, đồ bó gợi cảm) + **QR/Barcode**. Violence / logo: implement sau.

- **NSFW**: [`LukeJacob2023/nsfw-image-detector`](https://huggingface.co/LukeJacob2023/nsfw-image-detector) — ViT 5 lớp `drawings/hentai/neutral/porn/sexy`, score = `sexy+porn+hentai`. Public, nhẹ ~340MB, chạy CPU tốt.
- **QR/Barcode**: `pyzbar` (decode trực tiếp, không AI) — có mã đọc được → vi phạm.

**Chạy trên server/local:** notebook tự tạo `.venv` trong folder project, cài đặt + cache model vào đó — KHÔNG đụng Python hệ thống / ổ C.

Thứ tự: chạy **cell Bootstrap** → đổi kernel sang *Python (moderation)* → chạy các cell còn lại. Đặt ảnh test vào `sample/`.

## 0. Bootstrap — tạo venv trong folder & cài deps (chạy 1 lần, kernel nào cũng được)

In [ ]:
import os, sys, subprocess, pathlib

PROJ = pathlib.Path.cwd()
if PROJ.name == "notebooks":
    PROJ = PROJ.parent                       # chạy được dù mở từ notebooks/ hay root
VENV = PROJ / ".venv"
PYEXE = VENV / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

if not PYEXE.exists():
    print("Tạo venv:", VENV)
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)

REQ = ["transformers>=4.50.0", "accelerate", "pillow<12",
       "pandas", "requests", "ipykernel", "pyzbar"]
print("Cài deps vào", VENV, "(lần đầu hơi lâu)...")
subprocess.run([str(PYEXE), "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([str(PYEXE), "-m", "pip", "install", "-q", *REQ], check=True)
# torch: bản CPU cho nhẹ (~500MB). Server có GPU mới + driver CUDA 12.x thì đổi sang cu126/cu128.
subprocess.run([str(PYEXE), "-m", "pip", "install", "-q", "torch",
                "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([str(PYEXE), "-m", "ipykernel", "install", "--user",
                "--name", "moderation", "--display-name", "Python (moderation)"], check=True)
print("\nXONG. Đổi kernel: Kernel > Change Kernel > 'Python (moderation)', rồi chạy từ cell 1.")
# Lưu ý pyzbar:
#   - Windows: wheel đã kèm sẵn DLL zbar, không cần cài gì thêm.
#   - Linux:   cần thư viện hệ thống -> sudo apt-get install -y libzbar0

## 1. Cấu hình (chạy trong kernel *Python (moderation)*)

Đặt cache model vào folder project trước khi import transformers.

In [ ]:
import os, pathlib
PROJ = pathlib.Path.cwd()
if PROJ.name == "notebooks":
    PROJ = PROJ.parent
os.environ["HF_HOME"] = str(PROJ / ".hf_cache")   # weights tải về đây, không vào ~/.cache (C:)
IMAGE_DIR = str(PROJ / "sample")                  # thư mục ảnh test
print("HF_HOME  :", os.environ["HF_HOME"])
print("IMAGE_DIR:", IMAGE_DIR)

## 2. Load model NSFW (5 lớp)

In [ ]:
import torch
from transformers import (pipeline, AutoModelForImageClassification,
                          AutoImageProcessor, ViTImageProcessor)

NSFW_MODEL = "LukeJacob2023/nsfw-image-detector"
_device = 0 if torch.cuda.is_available() else -1

_mdl = AutoModelForImageClassification.from_pretrained(NSFW_MODEL)
try:
    _ip = AutoImageProcessor.from_pretrained(NSFW_MODEL)
except Exception:
    _ip = ViTImageProcessor.from_pretrained(NSFW_MODEL)  # repo thiếu image_processor_type
nsfw_clf = pipeline("image-classification", model=_mdl, image_processor=_ip, device=_device)
print("NSFW labels:", list(_mdl.config.id2label.values()), "| device:", _device)

## 3. Hàm moderation

In [ ]:
from PIL import Image
from pyzbar.pyzbar import decode as zbar_decode
import requests, io

_HEADERS = {"User-Agent": "Mozilla/5.0"}

def load_image(src):
    if isinstance(src, Image.Image):
        return src.convert("RGB")
    if str(src).startswith("http"):
        r = requests.get(src, headers=_HEADERS, timeout=30); r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    return Image.open(src).convert("RGB")

# ---------- QR / Barcode (pyzbar, không dùng AI) ----------
# Thử nhiều scale để bắt mã nhỏ/mờ; grayscale giúp zbar đọc tốt hơn. Tìm thấy là dừng.
CODE_SCALES = (1.0, 1.5, 2.0)

def detect_codes(img):
    """Trả về list mã đọc được: [{'type': 'QRCODE'|'EAN13'|..., 'data': '...'}]."""
    g = img.convert("L")
    found = {}
    for s in CODE_SCALES:
        im = g if s == 1.0 else g.resize((round(g.width * s), round(g.height * s)))
        for d in zbar_decode(im):
            found[(d.type, bytes(d.data))] = None
        if found:
            break
    return [{"type": t, "data": data.decode("utf-8", "replace")} for (t, data) in found]

# ---------- NSFW ----------
UNSAFE_LABELS = {"porn", "hentai", "sexy"}   # 'sexy' bắt bikini/đồ bó; bỏ neutral/drawings
ALL_LABELS = ["neutral", "drawings", "sexy", "porn", "hentai"]
NSFW_THR = 0.5

def nsfw_breakdown(img):
    return {p["label"].lower(): float(p["score"]) for p in nsfw_clf(img)}

# ---------- Tổng hợp (QR/barcode TRƯỚC, dừng sớm) ----------
# Chỉ cần kết quả cuối -> mã quét rẻ + chắc chắn nên check trước; có mã là REJECT luôn,
# bỏ qua bước NSFW tốn CPU. Ảnh bị mã loại sẽ KHÔNG có điểm NSFW (nsfw=None).
def moderate(src):
    img = load_image(src)
    codes = detect_codes(img)
    if codes:
        return {"codes": codes, "nsfw": None,
                "verdict": "REJECT", "violations": "qr_barcode"}
    nb = nsfw_breakdown(img)
    nsfw = sum(nb.get(k, 0.0) for k in UNSAFE_LABELS)
    return {**{k: nb.get(k, 0.0) for k in ALL_LABELS},
            "codes": [], "nsfw": nsfw,
            "verdict": "REJECT" if nsfw >= NSFW_THR else "ACCEPT",
            "violations": "nsfw" if nsfw >= NSFW_THR else ""}

## 4. Test 1 ảnh

In [ ]:
TEST_IMAGE = os.path.join(IMAGE_DIR, "sample-caught-01.jpg")   # đổi path hoặc URL
r = moderate(TEST_IMAGE)
print(r["verdict"], "|", r["violations"] or "-")
print("nsfw =", None if r["nsfw"] is None else round(r["nsfw"], 3), "| codes =", r["codes"])

## 5. Quét cả folder ảnh

In đủ điểm 5 lớp + tổng `nsfw` + verdict. Xem cột `sexy`/`neutral` để hiểu lý do và dò
threshold (nên có cả ảnh vi phạm LẪN ảnh sạch trong folder).

In [ ]:
import pandas as pd, glob, time

def _r(v):  # round an toàn (None -> None)
    return None if v is None else round(v, 3)

paths = sorted(p for e in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp")
               for p in glob.glob(os.path.join(IMAGE_DIR, e)))
print(f"Quét {len(paths)} ảnh trong {IMAGE_DIR}\n")

rows, t0 = [], time.time()
for i, p in enumerate(paths, 1):
    r = moderate(p)
    row = {"file": os.path.basename(p)}
    row.update({k: _r(r.get(k)) for k in ALL_LABELS})   # None nếu bị mã loại sớm
    row["nsfw"] = _r(r["nsfw"])
    row["code_types"] = ",".join(sorted({c["type"] for c in r["codes"]}))
    row["verdict"] = r["verdict"]
    row["violations"] = r["violations"]
    rows.append(row)
    print(f"[{i}/{len(paths)}] {row['file']:40} {row['verdict']:7} {row['violations']}")

df = pd.DataFrame(rows).sort_values(["verdict", "nsfw"], ascending=[True, False], na_position="first")
print(f"\nXong {len(paths)} ảnh / {time.time()-t0:.1f}s | REJECT: {(df['verdict']=='REJECT').sum()}")
df